# Блок 3. Практика: Организация хранения данных

В этом notebook мы:
1. Проанализируем производительность сложных запросов для поиска индикаторов атаки
2. Создадим оптимизированные индексы и измерим ускорение
3. Сравним производительность PostgreSQL (с индексами) и DuckDB (Parquet)
4. Выполним комплексный анализ инцидента, объединяя данные из всех источников

## Подготовка

Перед началом работы убедитесь, что:

1. **PostgreSQL запущен** (из Блока 1 или Блока 2)
2. **Данные загружены** в PostgreSQL через Vector (Блок 2) или через `load_data.py` (Блок 1)
3. **Parquet-файлы скачаны** в директорию `data/lite/` (через `bootstrap.py`)

Период атаки в данных: **дни 61-74** (14 дней). Все запросы должны покрывать этот период для поиска индикаторов атаки.

### Важно: Различия в схемах таблиц

В зависимости от способа загрузки данных, схема таблиц может отличаться:

- **Блок 1 (load_data.py)**: Данные загружаются напрямую из Parquet → колонки соответствуют схеме Parquet-файлов (без `severity`)
- **Блок 2 (Vector)**: Данные обрабатываются Vector → добавляется колонка `severity` на основе анализа логов

Миграция `sql/migrations/001_create_indexes.sql` автоматически адаптирует схему, добавляя колонку `severity` если её нет, и заполняет её значениями на основе существующих данных. После этого создаются оптимизированные индексы.

In [ ]:
import psycopg2
import duckdb

In [ ]:
# Параметры подключения к PostgreSQL
PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "security_logs",
    "user": "analyst",
    "password": "security123"  # Замените на ваш пароль из .env
}

# Строка подключения для DuckDB
PG_CONNECTION_STRING = f"postgresql://{PG_CONFIG['user']}:{PG_CONFIG['password']}@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"

## Задание 3.1: Анализ производительности запросов до индексов

Перед созданием индексов нужно понять, какие запросы работают медленно. Выполним 5 сложных аналитических запросов для поиска индикаторов атаки и замерим их производительность.

**Важно:** Все запросы должны покрывать период атаки - дни 61-74. Используйте `EXPLAIN ANALYZE` для анализа плана выполнения.

In [ ]:
# Подключаемся к PostgreSQL
conn = psycopg2.connect(**PG_CONFIG)
cursor = conn.cursor()

# Проверяем количество записей
cursor.execute("SELECT COUNT(*) FROM auth_events")
count = cursor.fetchone()[0]
print(f"Записей в таблице: {count:,}")

In [ ]:
# Смотрим структуру таблицы
cursor.execute("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'auth_events'
    ORDER BY ordinal_position
""")

print("Структура таблицы auth_events:")
print("-" * 50)
for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:<25} {'NULL' if row[2] == 'YES' else 'NOT NULL'}")

In [ ]:
# Смотрим примеры данных
cursor.execute("SELECT * FROM auth_events ORDER BY timestamp DESC LIMIT 5")
columns = [desc[0] for desc in cursor.description]

print("Примеры записей:")
for row in cursor.fetchall():
    print("-" * 50)
    for col, val in zip(columns, row):
        print(f"{col}: {val}")

In [ ]:
# Проверяем существующие индексы
cursor.execute("""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE tablename = 'auth_events'
""")

print("Существующие индексы:")
for row in cursor.fetchall():
    print(f"\n{row[0]}:")
    print(f"  {row[1]}")

### Вспомогательная функция для EXPLAIN ANALYZE

`EXPLAIN ANALYZE` показывает план выполнения запроса и реальное время выполнения. Это поможет понять, какие операции медленные и используются ли индексы.

In [ ]:
def explain_analyze(cursor, query):
    """Выполняет EXPLAIN ANALYZE и выводит результат."""
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    print("План выполнения:")
    print("=" * 80)
    for row in cursor.fetchall():
        print(row[0])
    print("=" * 80)

In [ ]:
# Запрос 1: Обнаружение brute-force атак
# TODO: Напишите запрос для поиска TOP-10 IP-адресов с наибольшим количеством 
# неудачных попыток входа за период атаки (дни 61-74).
# Для каждого IP покажите:
# - Количество попыток
# - Список атакуемых пользователей (через STRING_AGG)
# - Временной диапазон атаки (MIN и MAX timestamp)
# Отфильтруйте только те IP, у которых было более 50 неудачных попыток

query1 = """
-- Ваш запрос здесь
-- Подсказка: используйте WHERE timestamp >= '2024-03-01' AND timestamp < '2024-03-15'
-- для периода атаки (дни 61-74)
"""

print("Запрос 1: Обнаружение brute-force атак")
explain_analyze(cursor, query1)

In [ ]:
# Запрос 2: Обнаружение компрометации аккаунтов
# TODO: Напишите запрос для поиска пользователей, у которых был успешный вход 
# после серии неудачных попыток (более 5).
# Покажите:
# - Имя пользователя
# - IP-адрес, с которого произошёл успешный вход
# - Количество неудачных попыток перед успехом
# - Временную разницу между последней неудачей и успехом
# Отсортируйте по времени между неудачей и успехом (чем меньше, тем подозрительнее)

query2 = """
-- Ваш запрос здесь
-- Подсказка: используйте оконные функции LAG() или подзапросы для поиска 
-- последовательности неудач перед успехом
"""

print("Запрос 2: Обнаружение компрометации аккаунтов")
explain_analyze(cursor, query2)

In [ ]:
# Запрос 3: Обнаружение сканирования портов
# TODO: Напишите запрос для поиска IP-адресов, которые пытались подключиться 
# к более чем 10 различным портам за период атаки.
# Покажите:
# - IP-адрес источника
# - Количество уникальных портов
# - Список портов (через STRING_AGG)
# - Общее количество блокированных соединений
# Отфильтруйте только блокированные соединения (action = 'BLOCK')

query3 = """
-- Ваш запрос здесь
-- Используйте таблицу firewall_events
"""

print("Запрос 3: Обнаружение сканирования портов")
explain_analyze(cursor, query3)

In [ ]:
# Запрос 4: Обнаружение подозрительных веб-активностей
# TODO: Напишите запрос для поиска IP-адресов с наибольшим количеством 
# ошибок 403 и 404 за период атаки.
# Покажите:
# - IP-адрес
# - Количество ошибок 403
# - Количество ошибок 404
# - Список атакуемых путей (TOP-5 через подзапрос)
# - User-agent
# Отфильтруйте только те IP, у которых было более 100 ошибок

query4 = """
-- Ваш запрос здесь
-- Используйте таблицу nginx_logs
"""

print("Запрос 4: Обнаружение подозрительных веб-активностей")
explain_analyze(cursor, query4)

In [ ]:
# Запрос 5: Обнаружение подозрительных DNS-запросов
# TODO: Напишите запрос для поиска доменов, к которым было более 1000 запросов 
# за период атаки.
# Покажите:
# - Домен
# - Количество запросов
# - Список IP-адресов, которые делали запросы (TOP-5)
# - Типы запросов (A, AAAA, MX и т.д.)
# Отсортируйте по количеству запросов (подозрительные домены могут указывать на C2-каналы)

query5 = """
-- Ваш запрос здесь
-- Используйте таблицу dns_queries
"""

print("Запрос 5: Обнаружение подозрительных DNS-запросов")
explain_analyze(cursor, query5)

In [ ]:
# Сохраняем время выполнения запросов ДО создания индексов
import time

def measure_query_time(cursor, query, name):
    """Измеряет время выполнения запроса."""
    start = time.perf_counter()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = time.perf_counter() - start
    print(f"{name}: {elapsed*1000:.2f} мс ({len(results)} строк)")
    return elapsed, results

print("\n" + "="*80)
print("ВРЕМЯ ВЫПОЛНЕНИЯ ЗАПРОСОВ ДО СОЗДАНИЯ ИНДЕКСОВ")
print("="*80)

times_before = {}
times_before['query1'], _ = measure_query_time(cursor, query1, "Запрос 1: Brute-force")
times_before['query2'], _ = measure_query_time(cursor, query2, "Запрос 2: Компрометация")
times_before['query3'], _ = measure_query_time(cursor, query3, "Запрос 3: Сканирование портов")
times_before['query4'], _ = measure_query_time(cursor, query4, "Запрос 4: Веб-атаки")
times_before['query5'], _ = measure_query_time(cursor, query5, "Запрос 5: DNS-запросы")

## Задание 3.2: Создание оптимизированных индексов

Теперь создадим индексы для ускорения запросов. Используйте готовую миграцию `sql/migrations/001_create_indexes.sql`.

**Важно:** Миграция автоматически адаптирует схему таблиц, добавляя колонку `severity` если её нет (для данных из Блока 1), и заполняет её значениями на основе существующих данных. После этого создаются оптимизированные индексы.

### Применение миграции с индексами

TODO: Примените SQL-миграцию `sql/migrations/001_create_indexes.sql` к базе данных.

Подсказка: можете прочитать файл и выполнить SQL команды, или использовать psql напрямую.

In [ ]:
# TODO: Примените миграцию с индексами
# Вариант 1: Чтение файла и выполнение
# with open('sql/migrations/001_create_indexes.sql', 'r') as f:
#     migration_sql = f.read()
#     cursor.execute(migration_sql)
#     conn.commit()

# Вариант 2: Использование psql через subprocess
# import subprocess
# subprocess.run(['psql', '-h', 'localhost', '-U', 'analyst', '-d', 'security_logs', '-f', 'sql/migrations/001_create_indexes.sql'])

print("Миграция применена! Проверяем созданные индексы...")

# Проверяем созданные индексы
cursor.execute("""
    SELECT schemaname, tablename, indexname 
    FROM pg_indexes 
    WHERE schemaname = 'public' 
    AND tablename IN ('auth_events', 'nginx_logs', 'dns_queries', 'firewall_events')
    ORDER BY tablename, indexname
""")

print("\nСозданные индексы:")
for row in cursor.fetchall():
    print(f"  {row[1]}.{row[2]}")

# Обновляем статистику для оптимизатора
cursor.execute("ANALYZE auth_events, nginx_logs, dns_queries, firewall_events")
conn.commit()
print("\nСтатистика обновлена!")

### Сравнение производительности после создания индексов

Теперь выполним те же запросы и сравним производительность.

TODO: Выполните все 5 запросов повторно и сравните время выполнения.

In [ ]:
# Выполняем запросы после создания индексов
print("\n" + "="*80)
print("ВРЕМЯ ВЫПОЛНЕНИЯ ЗАПРОСОВ ПОСЛЕ СОЗДАНИЯ ИНДЕКСОВ")
print("="*80)

times_after = {}
times_after['query1'], _ = measure_query_time(cursor, query1, "Запрос 1: Brute-force")
times_after['query2'], _ = measure_query_time(cursor, query2, "Запрос 2: Компрометация")
times_after['query3'], _ = measure_query_time(cursor, query3, "Запрос 3: Сканирование портов")
times_after['query4'], _ = measure_query_time(cursor, query4, "Запрос 4: Веб-атаки")
times_after['query5'], _ = measure_query_time(cursor, query5, "Запрос 5: DNS-запросы")

# Сравнение производительности
print("\n" + "="*80)
print("СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ")
print("="*80)
print(f"{'Запрос':<35} {'До индексов (мс)':>20} {'После (мс)':>20} {'Ускорение':>15}")
print("-"*80)

for qname, label in [
    ('query1', 'Запрос 1: Brute-force'),
    ('query2', 'Запрос 2: Компрометация'),
    ('query3', 'Запрос 3: Сканирование портов'),
    ('query4', 'Запрос 4: Веб-атаки'),
    ('query5', 'Запрос 5: DNS-запросы')
]:
    before = times_before.get(qname, 0) * 1000
    after = times_after.get(qname, 0) * 1000
    speedup = before / after if after > 0 else 0
    print(f"{label:<35} {before:>20.2f} {after:>20.2f} {speedup:>14.2f}x")

print("="*80)

## Задание 3.3: Аналитика в DuckDB с Parquet-файлами

Для сложной аналитики по большим объёмам данных используйте DuckDB с готовыми Parquet-файлами из директории `data/lite/`.

**Важно:** Используйте готовые Parquet-файлы, не экспортируйте данные из PostgreSQL!

In [ ]:
# Подключаемся к DuckDB
duckdb_conn = duckdb.connect()

# TODO: Прочитайте Parquet-файлы для всех 4 типов логов за период атаки (дни 61-74)
# Подсказка: используйте read_parquet() с массивом путей или glob-паттернами
# Пример:
# auth_data = duckdb_conn.execute("""
#     SELECT * FROM read_parquet([
#         'data/lite/auth_events/day=61/*.parquet',
#         'data/lite/auth_events/day=62/*.parquet',
#         ...
#     ])
# """).df()

print("Parquet-файлы загружены в DuckDB")

In [ ]:
# TODO: Выполните те же 5 запросов через DuckDB, читая Parquet-файлы
# Адаптируйте запросы под синтаксис DuckDB (FROM первым, GROUP BY ALL и т.д.)

def measure_duckdb_parquet(duckdb_conn, query, name):
    """Измеряет время выполнения запроса в DuckDB по Parquet."""
    start = time.perf_counter()
    result = duckdb_conn.execute(query).fetchall()
    elapsed = time.perf_counter() - start
    print(f"{name}: {elapsed*1000:.2f} мс ({len(result)} строк)")
    return elapsed

print("\n" + "="*80)
print("ВРЕМЯ ВЫПОЛНЕНИЯ ЗАПРОСОВ В DUCKDB (PARQUET)")
print("="*80)

duckdb_times = {}

# Запрос 1: Brute-force через DuckDB
# TODO: Адаптируйте query1 для DuckDB синтаксиса
duckdb_query1 = """
-- Ваш запрос здесь
-- Используйте read_parquet() для чтения auth_events
"""

duckdb_times['query1'] = measure_duckdb_parquet(duckdb_conn, duckdb_query1, "Запрос 1: Brute-force")

# TODO: Выполните остальные 4 запроса аналогично

### Итоговое сравнение производительности

TODO: Составьте итоговую таблицу сравнения производительности всех трёх подходов:
1. PostgreSQL без индексов
2. PostgreSQL с индексами  
3. DuckDB (Parquet)

In [ ]:
# TODO: Составьте итоговую таблицу сравнения
print("\n" + "="*100)
print("ИТОГОВОЕ СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ (мс)")
print("="*100)
print(f"{'Запрос':<35} {'PG (без индексов)':>20} {'PG (с индексами)':>20} {'DuckDB (Parquet)':>20} {'Ускорение':>15}")
print("-"*100)

for qname, label in [
    ('query1', 'Запрос 1: Brute-force'),
    ('query2', 'Запрос 2: Компрометация'),
    ('query3', 'Запрос 3: Сканирование портов'),
    ('query4', 'Запрос 4: Веб-атаки'),
    ('query5', 'Запрос 5: DNS-запросы')
]:
    before = times_before.get(qname, 0) * 1000
    after = times_after.get(qname, 0) * 1000
    duckdb_time = duckdb_times.get(qname, 0) * 1000
    speedup_vs_before = before / duckdb_time if duckdb_time > 0 else 0
    
    print(f"{label:<35} {before:>20.2f} {after:>20.2f} {duckdb_time:>20.2f} {speedup_vs_before:>14.2f}x")

print("="*100)

# TODO: Сделайте выводы о применимости каждого подхода
print("\nВыводы:")
print("- PostgreSQL без индексов: ...")
print("- PostgreSQL с индексами: ...")
print("- DuckDB (Parquet): ...")

## Задание 3.4: Комплексный анализ инцидента

Выполните комплексный анализ, объединяющий данные из всех 4 источников для построения картины атаки.

### Корреляция событий

TODO: Найдите IP-адреса, которые появляются одновременно в нескольких типах логов (auth, nginx, firewall, dns).
Определите временную последовательность событий для подозрительных IP и постройте timeline атаки.

In [ ]:
# TODO: Найдите IP-адреса, присутствующие во всех типах логов
# Подсказка: используйте INTERSECT или JOIN для поиска общих IP

# Пример структуры запроса:
correlation_query = """
-- Найдите IP, которые есть в auth_events, nginx_logs, firewall_events и dns_queries
-- Покажите для каждого IP:
-- - Количество событий каждого типа
-- - Временной диапазон активности
-- - Типы обнаруженных атак
"""

print("Корреляция событий по IP-адресам:")
# Выполните запрос и выведите результаты

### Обнаружение этапов атаки

TODO: Определите этапы атаки для топ-3 самых подозрительных IP:

1. **RECON**: Сканирование портов (firewall_events с action=BLOCK)
2. **BRUTEFORCE**: Массовые неудачные попытки входа (auth_events с login_failure)
3. **COMPROMISE**: Успешный вход после неудач (auth_events)
4. **LATERAL**: Перемещение по сети (dns_queries к подозрительным доменам)

Постройте timeline атаки, показывающий последовательность этапов.

In [ ]:
# TODO: Постройте timeline атаки для топ-3 подозрительных IP
# Для каждого IP покажите:
# - Временную последовательность событий
# - Этапы атаки (RECON, BRUTEFORCE, COMPROMISE, LATERAL)
# - Ключевые индикаторы компрометации

timeline_query = """
-- Ваш запрос для построения timeline
-- Объедините данные из всех 4 таблиц по IP и времени
"""

print("Timeline атаки для подозрительных IP:")
# Выполните запрос и визуализируйте результаты (можно использовать pandas/matplotlib)

### Сравнение производительности комплексных запросов

TODO: Выполните комплексный запрос, объединяющий данные из всех 4 таблиц, в PostgreSQL (с индексами) и DuckDB (Parquet). Сравните время выполнения и сложность запроса.

In [ ]:
# TODO: Комплексный запрос для поиска полной картины атаки
# Объедините данные из всех 4 таблиц для построения полной картины инцидента

complex_query_pg = """
-- Ваш комплексный запрос для PostgreSQL
-- Найдите IP, которые:
-- 1. Делали неудачные попытки входа (auth_events)
-- 2. Получали ошибки 403/404 на веб-сервере (nginx_logs)
-- 3. Имели блокированные соединения (firewall_events)
-- 4. Делали запросы к подозрительным доменам (dns_queries)
-- Покажите полную картину атаки для каждого IP
"""

print("Комплексный анализ в PostgreSQL:")
pg_complex_time, _ = measure_query_time(cursor, complex_query_pg, "Комплексный запрос")

# TODO: Адаптируйте запрос для DuckDB
complex_query_duckdb = """
-- Ваш комплексный запрос для DuckDB
"""

print("\nКомплексный анализ в DuckDB:")
duckdb_complex_time = measure_duckdb_parquet(duckdb_conn, complex_query_duckdb, "Комплексный запрос")

print(f"\nУскорение DuckDB vs PostgreSQL: {pg_complex_time / duckdb_complex_time:.2f}x")

## Выводы

TODO: Сделайте выводы о применимости каждого подхода на основе выполненных заданий.

**PostgreSQL (OLTP)** хорош для:
- ...

**DuckDB (OLAP)** хорош для:
- ...

**Рекомендуемый workflow для FinanceFlow:**
1. Vector → PostgreSQL (свежие данные в реальном времени)
2. Готовые Parquet-файлы из S3 → DuckDB (аналитика исторических данных)
3. Индексы в PostgreSQL → ускорение типичных запросов в 5-10 раз

In [ ]:
# Закрываем соединения
cursor.close()
conn.close()
duckdb_conn.close()
print("Соединения закрыты")